# <u>Trending spot symmetry</u>


# 1) Importing modules

In [3]:
import numpy as np
# import pypyodbc
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import seaborn as sns
from matplotlib.colors import to_hex

# 2) Data import + process

In [4]:
data_path = r"../data/xlsx_exported_from_access/SpotPositionResults.xlsx" # dummy datapath

df = pd.read_excel(data_path) # reading xlsx as df

basic_info = ["ADate", "MachineName", "Energy", "Device", "Gantry Angle", "Spot"] # basic spot info
grad_ratio = ["hor_rt_gradient", "hor_lt_gradient", "vert_rt_gradient", "vert_lt_gradient", "bltr_rt_gradient", "bltr_lt_gradient", "tlbr_rt_gradient", "tlbr_lt_gradient"] # gradient info of the spots

sub_df = df[basic_info + grad_ratio].copy() # new filtered df only with gradient data

## Setup - Calculate gradient ratio (GR) for each profile

In [5]:
# pixel coordinates of the expected spot position
pred_xrv4000 = {'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175], \
                'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125], \
                'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]}

# assigning pixel coordinates info
sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

# gradient ratio (-1 is when the spot is perfectly symmetrical)
sub_df['gr_hor'] = sub_df["hor_rt_gradient"] / sub_df["hor_lt_gradient"] # horizontal gradient ratio
sub_df['gr_vert']  = sub_df["vert_rt_gradient"] / sub_df["vert_lt_gradient"] # vertical gradient ratio
sub_df['gr_bltr']  = sub_df["bltr_rt_gradient"] / sub_df["bltr_lt_gradient"] # bottom-left to top-right gradient ratio
sub_df['gr_tlbr']  = sub_df["tlbr_rt_gradient"] / sub_df["tlbr_lt_gradient"] # top-left to bottom-right gradient ratio

## Setup - mean GR across all positions of the same energy

In [6]:
# mean df
avg_df = (
    sub_df
    .groupby(["ADate","Device", "MachineName", "Gantry Angle", "Energy"], as_index=False) # as_index makes sure there are entries in all info column
    .agg( # calculating means of all gradient ratio
        mean_gr_hor=("gr_hor", "mean"),
        mean_gr_vert=("gr_vert", "mean"),
        mean_gr_bltr=("gr_bltr", "mean"),
        mean_gr_tlbr=("gr_tlbr", "mean")
         )
)

# checking df
for column, content in avg_df.items():
    print(column)
#    for info, data in content.items():
#        print(info)

ADate
Device
MachineName
Gantry Angle
Energy
mean_gr_hor
mean_gr_vert
mean_gr_bltr
mean_gr_tlbr


Testing the avg_df works with selected_df in Savanna's graphing code

In [7]:
selected_df = avg_df[(avg_df["MachineName"]=="Gantry 2") & (avg_df["Device"] == "XRV-3000") & (avg_df['ADate'] >= "2025-01-01") & (avg_df["Energy"] == 70) &(avg_df["Gantry Angle"] == 180)] # arbitrary selections

#selected_df # checking df

# 3) Graphs

## Test 1.0 - Ploting the mean GR of all profiles in one gantry

- Starting from Savanna's base code
- Adding a time filter starting on YYYY/MM/DD and going back N months
- Changing the y intervals depends on the data range; 0.01 if it is smaller than 0.1, and 0.02 if it is larger than 0.1

In [11]:
def plotly_mean_grad_ratio(df, grad, gantry, device, energy, gantry_angle, end_date, n_months):
    '''
    Plot gradient ratio time series data using plotly module.

    plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)

    Input:
        df:             Pandas dataframe

        grad:           Column name of the dataframe in string (e.g. "gr_hor")

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)

        end_date:       The end date of the data in string (e.g. "2026-01-01")

        n_months:        Number of months you want to go back from the end date in integer

    Return:
        Interactive plotly graph.
    '''

     # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=grad,
        color_discrete_sequence= px.colors.qualitative.T10,
        title=f'{gantry} - Mean Gradient Ratio across all spot positions for {energy} MeV',
        width=800,
        height=500
    )
    
    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,                 # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                    # thinner connecting lines
                    ))

    # Y-axis limit
    y_min = np.floor(selected_df[grad].min().min() * 100) / 100 # rounding minimum number to the lowest 0.01
    y_max = np.ceil(selected_df[grad].max().max() * 100) / 100 # rounding maximum number to the highest 0.01

    if y_max - y_min > 0.1:
        steps = 0.02 # y intervals will be 0.02 if the range of the data is over 0.1
    else:
        steps = 0.01 # y intervals will be 0.01 if the range of the data is smaller than 0.1

    fig.update_yaxes(range=[y_min,y_max], dtick=steps)

    # Show plot
    fig.show()

    return

In [12]:
# plotting absolute y-pos, on Gantry 4, from XRV-4000 data, 100 MeV spot, Gantry angle = 0, in last 2 months

profiles = ["mean_gr_hor", "mean_gr_vert", "mean_gr_bltr", "mean_gr_tlbr"]

plotly_mean_grad_ratio(avg_df, profiles, "Gantry 4", "XRV-3000", 100, 0,"2026-01-01", 24)

## Test 1.1 - Ploting the GR of all positions in one gantry

- List of spot positions 

In [13]:
xrv3000_spot = ["Top-Left", "Top-Centre", "Top-Right", 
                "Left", "Centre", "Right", 
                "Bottom-Left", "Bottom-Centre", "Bottom-Right"]

xrv4000_spot = ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right",
                "Top-Left", "Top-Centre", "Top-Right", 
                "Left", "Centre", "Right", 
                "Bottom-Left", "Bottom-Centre", "Bottom-Right", 
                "Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]

postism_spot = ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right",
                "Top-Left", "Top-Centre", "Top-Right", 
                "Left", "Centre", "Right", 
                "Bottom-Left", "Bottom-Right", 
                "Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]

- List of colours for each profile

In [14]:
color_map = {
    "gr_hor": "blue",
    "gr_vert": "red",
    "gr_bltr": "green",
    "gr_tlbr": "orange"
}

- Making subplots for each position with all profiles

In [24]:
def plotly_all_grad_ratio(df, grad, gantry, device, energy, gantry_angle, end_date, n_months):
    '''
    Plot gradient ratio time series data using plotly module.

    plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)

    Input:
        df:             Pandas dataframe

        grad:           Column name of the dataframe in string (e.g. "gr_hor")

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)

        end_date:       The end date of the data in string (e.g. "2026-01-01")

        n_months:        Number of months you want to go back from the end date in integer

    Return:
        Interactive plotly graph.
    '''

     # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) & (df["Gantry Angle"] == gantry_angle)]

    if device == "XRV-3000":

        # Plot
        subplot_titles = [f"{pos}" for pos in xrv3000_spot] # subplot titles

        fig = make_subplots(rows=3, cols=3,subplot_titles=subplot_titles)

        # making subplot per position 
        for i, spot in enumerate(xrv3000_spot):

            # looping for row/col index
            total_cols = 3
            row = i // total_cols + 1
            col = i % total_cols + 1

            plot_df = selected_df[(selected_df["Spot"]==spot)]

            # plot differnt profiles in each position 
            for profile in grad:

                fig.add_trace(
                    go.Scatter(
                        x=plot_df['ADate'], 
                        y=plot_df[profile], 
                        name=profile,
                        legendgroup=profile,
                        showlegend=(i==0),
                        line=dict(color=color_map[profile])
                        ), 
                        row=row, 
                        col=col
                    )


        # plot settings
        fig.update_layout(
            width=1280,
            height=720,
            margin=dict(l=50, r=50, t=50, b=50),
        )

        # Optional: connect points by spot for clarity
        fig.update_traces(mode='markers+lines',
                        marker=dict(size=4,                 # larger size
                                    line=dict(width=2)),     # outline width
                        line=dict(width=1                    # thinner connecting lines
                        ))


        fig.show()

    return

In [25]:
profiles = ["gr_hor", "gr_vert", "gr_bltr", "gr_tlbr"]

plotly_all_grad_ratio(sub_df, profiles, "Gantry 1", "XRV-3000", 100, 0,"2026-01-01", 12)

## Test 1.2 - Ploting all position for each GR in one gantry

- List of profiles

In [ ]:
profiles = ["mean_gr_hor", "mean_gr_vert", "mean_gr_bltr", "mean_gr_tlbr"]

In [ ]:
def plotly_all_grad_ratio(df, grad, gantry, device, energy, gantry_angle, end_date, n_months):
    '''
    Plot gradient ratio time series data using plotly module.

    plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)

    Input:
        df:             Pandas dataframe

        grad:           Column name of the dataframe in string (e.g. "gr_hor")

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)

        end_date:       The end date of the data in string (e.g. "2026-01-01")

        n_months:        Number of months you want to go back from the end date in integer

    Return:
        Interactive plotly graph.
    '''

     # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) & (df["Gantry Angle"] == gantry_angle)]


    # Plot
    subplot_titles = [f"{sym}" for sym in profiles] # subplot titles

    fig = make_subplots(rows=2, cols=2,subplot_titles=subplot_titles)

    # making subplot per position 
    for i, sym in enumerate(profiles):

        # looping for row/col index
        total_cols = 2
        row = i // total_cols + 1
        col = i % total_cols + 1

        plot_df = selected_df[(selected_df[sym])]

        # plot differnt profiles in each position 
        for spot in grad:

            fig.add_trace(
                go.Scatter(
                    x=plot_df['ADate'], 
                    y=plot_df[profile], 
                    name=profile,
                    legendgroup=profile,
                    showlegend=(i==0),
                    line=dict(color=color_map[profile])
                    ), 
                    row=row, 
                    col=col
                )


        # plot settings
        fig.update_layout(
            width=1280,
            height=720,
            margin=dict(l=50, r=50, t=50, b=50),
        )

        # Optional: connect points by spot for clarity
        fig.update_traces(mode='markers+lines',
                        marker=dict(size=4,                 # larger size
                                    line=dict(width=2)),     # outline width
                        line=dict(width=1                    # thinner connecting lines
                        ))


        fig.show()

    return